In [1]:
import random
import numpy as np

cars_served_rate = 0.3
green_time = {
    'fixed': 30,  # Fixed green light time for straight traffic
    'fixed_left': 10,  # Fixed green time for left turn
}
fixed_cars_served = {
    'straight': cars_served_rate * green_time['fixed'],  # number of cars that gets served for a given green time duration
    'left': cars_served_rate * green_time['fixed_left']
}
adaptive_cars_served = {
    'straight': (cars_served_rate) * green_time['fixed'],
    'left': cars_served_rate * green_time['fixed_left'],
    'total': cars_served_rate * green_time['fixed'] + cars_served_rate * green_time['fixed_left']
}

class TrafficLight:
    def __init__(self, direction, destination_lane):
        self.direction = direction  # e.g., "north-south" or "east-west"
        self.destination_lane = destination_lane
        self.queue_straight = 0
        self.queue_left = 0
        self.total_waiting_time = 0
        self.total_cars_served = 0
        self.total_cars_generated = 0
        self.cycle_queue_lengths = []  # Track queue lengths per cycle
        self.queue_tot = 0


        
    def generate_traffic(self,seed,mode,traffic_type):
        # Randomly generate cars arriving at each queue
        if traffic_type == 'rush_hour':
            straight_range = (10,30) if mode == 'fixed' else (15,40)
            left_range = (5,15) if mode == 'fixed' else (0,0)

        elif traffic_type == "low_traffic":
            straight_range = (0, 5) if mode == "fixed" else (0, 7)
            left_range = (0, 3) if mode == "fixed" else (0, 0)

        else:
            straight_range = (0, 15) if mode == "fixed" else (0, 20)
            left_range = (0, 10) if mode == "fixed" else (0, 0)
            

        
        random.seed(seed)
        arrivals_straight = random.randint(*straight_range) 
        arrivals_left = random.randint(*left_range) if mode == 'fixed' else 0
        self.queue_straight += arrivals_straight
        self.queue_left += arrivals_left
        self.total_cars_generated += arrivals_straight + arrivals_left    
        
    def serve_traffic(self, mode, direction, traffic_type):
        total_queue = 0
        if mode == 'fixed':
            if traffic_type == 'rush_hour':
                served = fixed_cars_served
                green_left_time = green_time['fixed_left'] * 2
                green_straight_time = green_time['fixed'] * 2
            else:
                served = fixed_cars_served
                green_left_time = green_time['fixed_left']
                green_straight_time = green_time['fixed']
        else:  # Adaptive mode with dynamic timing
            # Dynamically adjust green time for straight traffic
            if traffic_type == 'normal':
                if self.queue_straight >= adaptive_cars_served['total'] :
                    green_straight_time = random.randint(40, 60)
                elif self.queue_straight < adaptive_cars_served['total']:
                    green_straight_time = random.randint(5, 15)
                elif self.queue_straight == 0:
                    green_straight_time = 0
                else:
                    green_straight_time = random.randint(15,30)  # Default adaptive time
                
                # Dynamically adjust green time for left turns
                served = {
                    'straight': (cars_served_rate) * green_straight_time
                }
            elif traffic_type == 'rush_hour':
                max = 20
                low = 10
                if self.queue_straight >= max :
                    green_straight_time = random.randint(66, 80)
                elif self.queue_straight < low:
                    green_straight_time = random.randint(15, 30)
                elif self.queue_straight == 0:
                    green_straight_time = 0
                else:
                    green_straight_time = random.randint(30,60)  # Default adaptive time
                
                # Dynamically adjust green time for left turns
                served = {
                    'straight': (cars_served_rate) * green_straight_time
                }

            else:
                if self.queue_straight >= 5 :
                    green_straight_time = random.randint(16, 24)
                elif 1 < self.queue_straight < 5:
                    green_straight_time = random.randint(3, 14)
                elif self.queue_straight == 0:
                    green_straight_time = 0
                else:
                    green_straight_time = random.randint(15,25)  # Default adaptive time
                
                # Dynamically adjust green time for left turns
                served = {
                    'straight': (cars_served_rate) * green_straight_time
                }
                
                

        # print(arrivals_fixed.append(self.arrival_time))
        if mode == 'fixed':
            cars_left = min(self.queue_left, served['left']) 
            
            total_queue += self.queue_left
            self.queue_left -= cars_left
            self.total_cars_served += cars_left

        
        # self.total_waiting_time +=  green_left_time
        cars_straight = min(self.queue_straight, served['straight'])
        # Serve straight traffic
        total_queue += self.queue_straight
        self.queue_straight -= cars_straight
        self.total_cars_served += cars_straight

        # Update total waiting time for remaining cars in the queue
        if mode == 'fixed':
            self.total_waiting_time +=  green_straight_time   + green_left_time  

        else:
            self.total_waiting_time +=  green_straight_time 
            
        self.cycle_queue_lengths.append(total_queue)

    



def simulate_traffic(cycles, mode, north_south, east_west,seed, traffic_type):
    """Simulate traffic for a given number of cycles and mode."""
    
    for _ in range(cycles):
        # Generate new arrivals at both directions
        
        north_south_seed = seed * 2
        east_west_seed = seed * 3
        north_south.generate_traffic(north_south_seed,mode,traffic_type)
        east_west.generate_traffic(east_west_seed,mode,traffic_type)
        
        
        # Serve traffic for each direction
       
        north_south.serve_traffic(mode, 'north_south',traffic_type)
        east_west.serve_traffic(mode, 'east_west',traffic_type)

    # Compute metrics for the simulation
    metrics = {
        'total_cars_served': round(north_south.total_cars_served + east_west.total_cars_served),
        'average_waiting_time': (north_south.total_waiting_time/cycles + east_west.total_waiting_time/cycles) / 2,
        'average_queue_length': round((
            sum(north_south.cycle_queue_lengths) / cycles +
            sum(east_west.cycle_queue_lengths) / cycles
        ) / 2),
        'traffic_flow_efficiency': np.round((
            north_south.total_cars_served + east_west.total_cars_served
        ) / (north_south.total_cars_generated + east_west.total_cars_generated) * 100 , 2) ,
        
    }
    return metrics


def run_multiple_simulations(runs, cycles, mode,traffic_type):
    """Run multiple simulations and collect metrics."""
    results = []
    for run in range(runs):
        seed = run*10
        # Initialize traffic lights for both directions
        north_south = TrafficLight(direction="north-south", destination_lane="south")
        east_west = TrafficLight(direction="east-west", destination_lane="west")
        results.append(simulate_traffic(cycles, mode, north_south, east_west, seed, traffic_type))
    return results



def main(): 
    
    
    # Run simulations for 5 runs
    cycles = 10
    runs = 5
    fixed_results = run_multiple_simulations(runs, cycles, 'fixed', 'normal')
    adaptive_results = run_multiple_simulations(runs, cycles, 'adaptive', 'normal')
    
    fixed_rush_hour = run_multiple_simulations(runs, cycles, 'fixed', 'rush_hour')
    fixed_low_traffic = run_multiple_simulations(runs, cycles, 'fixed', 'low_traffic')
    
    adaptive_rush_hour = run_multiple_simulations(runs, cycles, 'adaptive', 'rush_hour')
    adaptive_low_traffic = run_multiple_simulations(runs, cycles, 'adaptive', 'low_traffic')
    
    
    
    # Print results for each run
    print("Fixed-Time Results:")
    for i, result in enumerate(fixed_results, 1):
        print(f"Run {i}: {result}")
    
    print("\nAdaptive-Time Results:")
    for i, result in enumerate(adaptive_results, 1):
        print(f"Run {i}: {result}")
    print("\nFixed-Time Rush Hour Results:")
    for i, result in enumerate(fixed_rush_hour, 1):
        print(f"Run {i}: {result}")
    
    print("\nFixed-Time Low Traffic Results:")
    for i, result in enumerate(fixed_low_traffic, 1):
        print(f"Run {i}: {result}")
    
    print("\nAdaptive-Time Rush Hour Results:")
    for i, result in enumerate(adaptive_rush_hour, 1):
        print(f"Run {i}: {result}")
    
    print("\nAdaptive-Time Low Traffic Results:")
    for i, result in enumerate(adaptive_low_traffic, 1):
        print(f"Run {i}: {result}")

    #metrics for analysis

    #import math
    
    # a = 'total_cars_served'
    # b = 'average_waiting_time'
    # c = 'average_queue_length'
    # d = 'traffic_flow_efficiency'
    
    # a1 = 'Total Cars Served'
    # b1 = 'Average Waiting Time'
    # c1 = 'Average Queue Length'
    # d1 = 'Traffic Flow Efficiency'
    
    # title = d1
    # val = d
    # result = fixed_results
    # comp = 'FIXED'
    # print(comp + " TIME COMPARISON")
    # print(title + ' Comparison')
    # total = 0
    # for i in range(5):
    
    #     total += result[i][val]
    
    # n = 5
    
    # mean = (1/n) * total
    
    # print(f"Mean: {np.round(mean,2)}")
    
    # sum_for_sd = 0
    
    # for i in range(5):
    #     sum_for_sd += (result[i][val] - mean)**2
        
    # standard_deviation = math.sqrt((1/(n-1)) * sum_for_sd)
    
    # standard_error = standard_deviation/math.sqrt(5)
    
    # print(f"Standard Deviation = {np.round(standard_deviation,2)}")
    # print(f"Standard Error = {np.round(standard_error,2)}")
    # print(arrivals_fixed_ew)
    # print(arrivals_adaptive_ew)
    
    # arrivals_fixed_ew_dict = {}
    # arrivals_adaptive_ew_dict = {}
    # arrivals_fixed_ns_dict = {}
    # arrivals_adaptive_ns_dict = {}

if __name__ == '__main__':
    main()
    










Fixed-Time Results:
Run 1: {'total_cars_served': 240, 'average_waiting_time': 40.0, 'average_queue_length': 45, 'traffic_flow_efficiency': np.float64(66.67)}
Run 2: {'total_cars_served': 190, 'average_waiting_time': 40.0, 'average_queue_length': 29, 'traffic_flow_efficiency': np.float64(73.08)}
Run 3: {'total_cars_served': 240, 'average_waiting_time': 40.0, 'average_queue_length': 45, 'traffic_flow_efficiency': np.float64(66.67)}
Run 4: {'total_cars_served': 190, 'average_waiting_time': 40.0, 'average_queue_length': 12, 'traffic_flow_efficiency': np.float64(95.0)}
Run 5: {'total_cars_served': 210, 'average_waiting_time': 40.0, 'average_queue_length': 19, 'traffic_flow_efficiency': np.float64(87.5)}

Adaptive-Time Results:
Run 1: {'total_cars_served': 240, 'average_waiting_time': 47.0, 'average_queue_length': 12, 'traffic_flow_efficiency': np.float64(100.0)}
Run 2: {'total_cars_served': 207, 'average_waiting_time': 36.0, 'average_queue_length': 12, 'traffic_flow_efficiency': np.float64(